In [ ]:
# !pip install git+https://github.com/eduardobatista/ActVibModules.git

  Cloning https://github.com/eduardobatista/ActVibModules.git to c:\users\samsung\appdata\local\temp\pip-req-build-p9cwixxf
  Resolved https://github.com/eduardobatista/ActVibModules.git to commit 9ed4b39a96cb63583a66ac3a48eb1709395881f6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/eduardobatista/ActVibModules.git 'C:\Users\Samsung\AppData\Local\Temp\pip-req-build-p9cwixxf'
You should consider upgrading via the 'c:\202502\IC\Codes\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import numpy as np
from scipy import signal
import plotly.express as px
import pandas as pd
from ActVibModules.DSPFuncs import easyFourier

In [3]:
Namostras = 5000
Nfiltro = 101

x = np.random.randn(Namostras) # Gerando um sinal de entrada de 10k amostras

# Gerando uma resposta a ser identificada
# (usei janela de hamming com 101 coefs, mas poderia ser qquer coisa):
wplanta = np.hamming(Nfiltro)

d = signal.lfilter(wplanta,1,x) # Obtendo o sinal desejado, filtrando x com wplanta

# Os sinais x e d devem ser passados para o filtro adaptativo e ele deve conseguir
# obter wplanta a partir desses sinais.

In [4]:
mu = 0.1
fi = 1e-3
xx = np.zeros(Nfiltro) # Vetor de entrada do adaptativo: tipo de buffer que deve armazenar as últimas Nfiltro amostras.
ww = np.zeros(Nfiltro) # Vetor de coeficientes do filtro adaptativo, inicializado com zeros.

# Vetor para salvar os erros ao quadrado somente para ver a convergência (não é preciso ser implementado no FPGA):
ee = np.zeros(Namostras)

for i in range(Namostras):
    # Primeiro o cálculo de y (saída do FIR adaptativo), que corresponde ao cálculo
    # da saída de um FIR com coeficientes ww:
    xx[1:] = xx[0:-1]
    xx[0] = x[i]
    y = xx @ ww
    # Agora o cálculo do erro:
    e = d[i] - y
    # Finalmente a adaptação:
    ww = ww + mu * e * xx / (xx @ xx + fi)
    # Salvando o erro ao quadrado para debug:
    ee[i] = e**2


In [5]:
# Se houver convergência, erro deve diminuir ao longo do tempo:
px.line(ee).show()

In [6]:
# Agora vamos ver se a resposta ao impulso do adaptativo chegou à do filtro de referência:
fig = px.line()
fig.add_scatter(y=wplanta,name='original')
fig.add_scatter(y=ww,name='adaptativo')
fig.show()

# Modelagem com Sinal Medido

In [7]:
data = pd.read_pickle("noise.pkl")
display(data)

,t_s,accel_1,accel_2,packed_u64,raw_u64,dac_signal
0,0.000006,-606,-1004,373622504468,373622504468,86
1,0.001250,-140,-460,6124614254132,6124614254132,1425
2,0.002458,511,-1173,7030895016811,7030895016811,1637
3,0.003657,1148,-488,15178489724440,15178489724440,3534
4,0.004857,2262,-64,9126953811904,9126953811904,2125
...,...,...,...,...,...,...
75869,89.994501,857,1258,15552132744426,15552132744426,3621
75870,89.995685,214,-380,12120411799172,12120411799172,2822
75871,89.996871,-367,548,360753201700,360753201700,83
75872,89.998057,-840,-1209,6077323737927,6077323737927,1414


In [8]:
x = data['dac_signal'].values.astype(float)
x = x - np.mean(x) # Removendo a média de x
d1 = data['accel_1'].values.astype(float)
d1 = d1 - np.mean(d1) # Removendo a média de d1
d2 = data['accel_2'].values.astype(float)
d2 = d2 - np.mean(d2) # Removendo a média de d2

In [9]:
mu = 0.1
fi = 1e-3
Nfiltro = 4000
Namostras = len(x)

xx = np.zeros(Nfiltro) # Vetor de entrada do adaptativo: tipo de buffer que deve armazenar as últimas Nfiltro amostras.
ww = np.zeros(Nfiltro) # Vetor de coeficientes do filtro adaptativo, inicializado com zeros.

# Vetor para salvar os erros ao quadrado somente para ver a convergência (não é preciso ser implementado no FPGA):
ee = np.zeros(Namostras)

for i in range(Namostras):
    # Primeiro o cálculo de y (saída do FIR adaptativo), que corresponde ao cálculo
    # da saída de um FIR com coeficientes ww:
    xx[1:] = xx[0:-1]
    xx[0] = x[i]
    y = xx @ ww
    # Agora o cálculo do erro:
    e = d1[i] - y # Ativar esse para modelar o caminho para o Accel_1
    # e = d2[i] - y # Ativar esse para modelar o caminho para o Accel_2
    # Finalmente a adaptação:
    ww = ww + mu * e * xx / (xx @ xx + fi)
    # Salvando o erro ao quadrado para debug:
    ee[i] = e**2

In [10]:
fig = px.line()
fig.add_scatter(y=ww,name='adaptativo')
fig.update_layout(title_text='Resposta ao Impulso')
fig.show()

mag,freq = easyFourier(ww,fs=833)
fig = px.line()
fig.add_scatter(x=freq,y=mag,name='magnitude')
fig.update_layout(title_text='Magnitude')
fig.show()

In [11]:
dados = {
    "wf": [32767, 0, 0, 0, 0, 0, 0, 0],
    "ws": [32767, 0, 0, 0, 0, 0, 0, 0],
}

pd.to_pickle(dados, "coeffs.pkl")

# Modelagem DAC 1 

In [12]:
data = pd.read_pickle("noise_dac1.pkl")
display(data)

,t_s,accel_1,accel_2,dac_1,dac_2,sample_counter,sample_valid,config_dac1,config_dac2,pending_wf,pending_ws,ready_wf,ready_ws,overrun_wf,overrun_ws,raw_export,raw_data
0,0.000005,-751,3664,2484,0,915884,0,3,0,0,0,0,0,0,0,43698994380082768,3029515187222608300
1,0.001281,-10,3576,1747,0,915885,0,3,0,0,0,0,0,0,0,30733553313910264,3029515187222608301
2,0.002520,268,1161,1694,0,915887,0,3,0,0,0,0,0,0,0,29801163176805513,3029515187222608303
3,0.003751,268,-252,2682,0,915888,0,3,0,0,0,0,0,0,0,47182242988752644,3029515187222608304
4,0.004977,-279,-1930,2536,0,915889,0,3,0,0,0,0,0,0,0,44613788085385334,3029515187222608305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73460,89.995095,2050,3056,1037,0,991264,0,3,0,0,0,0,0,0,0,18243097062411248,3029515187222683680
73461,89.996319,2993,4493,55,0,991265,0,3,0,0,0,0,0,0,0,967570428596621,3029515187222683681
73462,89.997541,2903,5859,444,0,991266,0,3,0,0,0,0,0,0,0,7810930793977571,3029515187222683682
73463,89.998764,1728,4894,1778,0,991267,0,3,0,0,0,0,0,0,0,31278906900222750,3029515187222683683


In [13]:
x = data['dac_1'].values.astype(float)
x = x - np.mean(x) # Removendo a média de x
d1 = data['accel_1'].values.astype(float)
d1 = d1 - np.mean(d1) # Removendo a média de d1
d2 = data['accel_2'].values.astype(float)
d2 = d2 - np.mean(d2) # Removendo a média de d2

In [14]:
mu = 0.1
fi = 1e-3
Nfiltro = 2000
Namostras = len(x)

xx = np.zeros(Nfiltro) # Vetor de entrada do adaptativo: tipo de buffer que deve armazenar as últimas Nfiltro amostras.
ww = np.zeros(Nfiltro) # Vetor de coeficientes do filtro adaptativo, inicializado com zeros.

# Vetor para salvar os erros ao quadrado somente para ver a convergência (não é preciso ser implementado no FPGA):
ee = np.zeros(Namostras)

for i in range(Namostras):
    # Primeiro o cálculo de y (saída do FIR adaptativo), que corresponde ao cálculo
    # da saída de um FIR com coeficientes ww:
    xx[1:] = xx[0:-1]
    xx[0] = x[i]
    y = xx @ ww
    # Agora o cálculo do erro:
    e = d1[i] - y # Ativar esse para modelar o caminho para o Accel_1
    # e = d2[i] - y # Ativar esse para modelar o caminho para o Accel_2
    # Finalmente a adaptação:
    ww = ww + mu * e * xx / (xx @ xx + fi)
    # Salvando o erro ao quadrado para debug:
    ee[i] = e**2

In [15]:
fig = px.line()
fig.add_scatter(y=ww,name='adaptativo')
fig.update_layout(title_text='Resposta ao Impulso')
fig.show()

mag,freq = easyFourier(ww,fs=833)
fig = px.line()
fig.add_scatter(x=freq,y=mag,name='magnitude')
fig.update_layout(title_text='Magnitude')
fig.show()

In [16]:
def float_array_to_q15(arr):
    """
    Converte numpy array com coeficientes float em [-1, 1]
    para signed int16 em Q1.15.
    """
    arr = np.asarray(arr, dtype=np.float64)

    q = np.round(arr * (2**15))

    q = np.clip(q, -32768, 32767)

    return q.astype(np.int16)

In [17]:
ws = float_array_to_q15(ww)
ws.max()

np.int16(2445)

# Modelagem DAC 2

In [18]:
data = pd.read_pickle("noise_dac2.pkl")
display(data)

,t_s,accel_1,accel_2,dac_1,dac_2,sample_counter,sample_valid,config_dac1,config_dac2,pending_wf,pending_ws,ready_wf,ready_ws,overrun_wf,overrun_ws,raw_export,raw_data
0,0.000005,1042,-292,0,1529,692372,0,0,3,0,0,0,0,0,0,6567073349340,2453336000089985172
1,0.001279,626,-1279,0,4042,692373,0,0,3,0,0,0,0,0,0,17360298900225,2453336000089985173
2,0.002518,546,-1742,0,3882,692374,0,0,3,0,0,0,0,0,0,16673098889522,2453336000089985174
3,0.003748,617,-1595,0,2386,692375,0,0,3,0,0,0,0,0,0,10247832467909,2453336000089985175
4,0.004973,1341,-1007,0,1355,692376,0,0,3,0,0,0,0,0,0,5819768634385,2453336000089985176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73499,89.994527,1545,2012,0,3919,767750,0,0,3,0,0,0,0,0,0,16832078088156,2453336000090060550
73500,89.995751,955,1742,0,3390,767752,0,0,3,0,0,0,0,0,0,14560001722062,2453336000090060552
73501,89.996974,955,1149,0,2544,767752,0,0,3,0,0,0,0,0,0,10926459389053,2453336000090060552
73502,89.998196,683,546,0,1984,767753,0,0,3,0,0,0,0,0,0,8521259876898,2453336000090060553


In [19]:
x = data['dac_2'].values.astype(float)
x = x - np.mean(x) # Removendo a média de x
d1 = data['accel_1'].values.astype(float)
d1 = d1 - np.mean(d1) # Removendo a média de d1
d2 = data['accel_2'].values.astype(float)
d2 = d2 - np.mean(d2) # Removendo a média de d2

In [20]:
mu = 0.1
fi = 1e-3
Nfiltro = 2000
Namostras = len(x)

xx = np.zeros(Nfiltro) # Vetor de entrada do adaptativo: tipo de buffer que deve armazenar as últimas Nfiltro amostras.
ww = np.zeros(Nfiltro) # Vetor de coeficientes do filtro adaptativo, inicializado com zeros.

# Vetor para salvar os erros ao quadrado somente para ver a convergência (não é preciso ser implementado no FPGA):
ee = np.zeros(Namostras)

for i in range(Namostras):
    # Primeiro o cálculo de y (saída do FIR adaptativo), que corresponde ao cálculo
    # da saída de um FIR com coeficientes ww:
    xx[1:] = xx[0:-1]
    xx[0] = x[i]
    y = xx @ ww
    # Agora o cálculo do erro:
    e = d1[i] - y # Ativar esse para modelar o caminho para o Accel_1
    # e = d2[i] - y # Ativar esse para modelar o caminho para o Accel_2
    # Finalmente a adaptação:
    ww = ww + mu * e * xx / (xx @ xx + fi)
    # Salvando o erro ao quadrado para debug:
    ee[i] = e**2

In [21]:
fig = px.line()
fig.add_scatter(y=ww,name='adaptativo')
fig.update_layout(title_text='Resposta ao Impulso')
fig.show()

mag,freq = easyFourier(ww,fs=833)
fig = px.line()
fig.add_scatter(x=freq,y=mag,name='magnitude')
fig.update_layout(title_text='Magnitude')
fig.show()

In [22]:
wf = float_array_to_q15(ww)

In [23]:
coeffs = {
    "wf": wf.astype(int).tolist(),
    "ws": ws.astype(int).tolist(),
}

pd.to_pickle(coeffs, "coeffs_2000.pkl")

In [15]:
n_target = 300

wf_full = np.pad(wf, (0, max(0, n_target - len(wf))), constant_values=0)
ws_full = np.pad(ws, (0, max(0, n_target - len(ws))), constant_values=0)

coeffs = {
    "wf": wf_full.astype(int).tolist(),
    "ws": ws_full.astype(int).tolist(),
}

In [16]:
print(len([v for v in coeffs["wf"] if v != 0]))
print(len(coeffs["wf"]))

300
300


In [17]:
pd.to_pickle(coeffs, "coeffs_300.pkl")

In [24]:
wf.shape

(2000,)